# HW2: continue our 60,000-step Transformer to 180,000 updates
Run phases separately, starting on CPU. Do not use Run all. The CPU phase verifies Drive files and exact equality of the old and optimized training tensors. The GPU phase performs a bounded throughput comparison, chooses a path only on measured evidence, and continues our own model. No outside examples, additional targets, pretrained weights or handwritten answer repairs are used.

Six checkpoints and all 1,919 exact predictions per checkpoint are saved to Drive before export and bot evaluation. This notebook pins source commit 681c74ee5816d4030cca3e7b51cd630f927b3ceb. The run name intentionally rejects duplicate launches.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU first'
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)

In [ ]:
import os, subprocess, sys
from pathlib import Path
assert not Path('/dev/nvidia0').exists()
repo=Path('/content/hw2-next'); assert not repo.exists(), 'Inspect existing preparation before repeating'
subprocess.run(['git','init',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--depth=1','https://github.com/OsVox/hse-nlp.git','681c74ee5816d4030cca3e7b51cd630f927b3ceb'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','FETCH_HEAD'],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
with open('/content/prepare-60k.log','w') as log:
    preparation=subprocess.Popen([sys.executable,'-u','-m','homeworks.hw2.extend_colab'],stdout=log,stderr=subprocess.STDOUT)
print('CPU_PREPARATION_PID',preparation.pid)

In [ ]:
print('CPU_PREPARATION_RETURN_CODE',preparation.poll())
print(Path('/content/prepare-60k.log').read_text()[-5000:])

## GPU phase: only after CPU_60K_CONTINUATION_READY and exit code 0
Select G4 (the measured training hardware). The source, checkpoint, benchmark pool and training Parquet files are restored from verified Drive copies. The GPU probe compares legacy, reused tokenization, and two-batch CPU prefetch in reversed rounds. A new path needs at least 5% median throughput improvement. Temporary probe weights are discarded.

The continuation retains the 60k optimizer state, uses batch 128, BF16, peak learning rate 0.0003, 500 warmup updates, and cosine decay to 10% of the peak. At 80k, 100k, 120k, 140k, 160k and 180k it saves the checkpoint and exact predictions to Drive. The process requests runtime release at success or failure; verify release in Runtime > Manage sessions because the request can fail.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=300000)
from pathlib import Path
import os,sys,json,hashlib,shutil,tarfile,subprocess
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
prepared=root/'prepared/continuation-60k-v1'
ready=json.loads((prepared/'ready.json').read_text())
assert ready['gpu_allocated'] is False
assert ready['source_commit']=='681c74ee5816d4030cca3e7b51cd630f927b3ceb'
repo=Path('/content/hw2-next-gpu');repo.mkdir(exist_ok=True)
bundle=Path('/content/next-source.tar');shutil.copyfile(prepared/'source.tar',bundle)
assert hashlib.sha256(bundle.read_bytes()).hexdigest()==ready['source_sha256']
with tarfile.open(bundle) as z:z.extractall(repo,filter='data')
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
assert not (root/'runs/continue-60k-180k-b128-v1').exists(), 'Inspect existing run before repeating'
with (prepared/'execution.log').open('w') as log:
    execution=subprocess.Popen([sys.executable,'-u','-m','homeworks.hw2.extend_colab','--gpu'],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('GPU_EXECUTION_PID',execution.pid)

In [ ]:
print('EXECUTION_RETURN_CODE',execution.poll())
print((prepared/'execution.log').read_text()[-3000:])
run=root/'runs/continue-60k-180k-b128-v1'
if (run/'training.log').exists():print((run/'training.log').read_text()[-1800:])
print('CHECKPOINTS_READY',[p.parent.name for p in sorted(run.glob('step-*/results.complete.json'))])
if (run/'execution.complete.json').exists():print((run/'execution.complete.json').read_text())

## Export and evaluate
After the run has finished and the GPU is released, switch to CPU, mount Drive and download the archive below. For intermediate evaluations, export the saved JSONL directly from its Drive checkpoint folder. Submit every checkpoint's file to @hse_nlp_hw2_2026_bot and record its SHA256, score, count and rank. Do not use failed bot examples as training targets or edit individual generated programs.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Export after releasing GPU'
from google.colab import drive,files
drive.mount('/content/drive',timeout_ms=300000)
run=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/continue-60k-180k-b128-v1')
print((run/'execution.complete.json').read_text())
files.download(str(run/'predictions-and-evidence.zip'))